In [54]:
import json, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from mpl_toolkits.mplot3d import Axes3D          # noqa
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import color, filters, exposure
from skimage.transform import resize as sk_resize
from skimage.filters import threshold_otsu
import tifffile
import gudhi
warnings.filterwarnings("ignore")

# Inputs

test = 5

TIFF_PATH    = f"/Users/chanuka/Desktop/codespaces/ph_cancer/magicScan_TDA_ML/Subset1/Subset1_Test_{test}.tiff"
GEOJSON_PATH = f"/Users/chanuka/Desktop/codespaces/ph_cancer/magicScan_TDA_ML/Subset1_annot/Subset1_Test_{test}.geojson"
OUT_PATH     = f"/Users/chanuka/Desktop/codespaces/ph_cancer/figure_{test}.png"

CROP_SIZE = 1024
GAUSS_SIGMA = 10

T_low = 60
T_high = 80

### Version 6

In [51]:
# dealing with ground truth

with open(GEOJSON_PATH) as fj:
    gj = json.load(fj)
features_gj = gj.get("features", [])

all_x, all_y = [], []
for feat in features_gj:
    geom  = feat.get("geometry", {})
    gtype = geom.get("type","")
    rings = ([geom["coordinates"][0]] if gtype=="Polygon"
             else [p[0] for p in geom["coordinates"]] if gtype=="MultiPolygon"
             else [])
    for ring in rings:
        pts = np.array(ring, dtype=float)
        if pts.ndim==2 and pts.shape[1]>=2:
            all_x.extend(pts[:,0]); all_y.extend(pts[:,1])

cx_ann = int((min(all_x)+max(all_x))/2)
cy_ann = int((min(all_y)+max(all_y))/2)


# loading TIFF file

raw = tifffile.imread(TIFF_PATH)
if raw.ndim==4: raw=raw[0]
if raw.ndim==3 and raw.shape[0] in (1,3,4): raw=np.moveaxis(raw,0,-1)
if raw.shape[-1]==4: raw=raw[...,:3]
if raw.dtype!=np.uint8: raw=(raw/raw.max()*255).astype(np.uint8)
H,W = raw.shape[:2]

half = CROP_SIZE//2
cy_c = max(half, min(H-half, cy_ann))
cx_c = max(half, min(W-half, cx_ann))
row0,row1 = cy_c-half, cy_c+half
col0,col1 = cx_c-half, cx_c+half
img_rgb = raw[row0:row1, col0:col1]


# Conversion to Haematoxylin-Eosin-DAB (HED) color space
hed      = color.rgb2hed(img_rgb)
hema_raw = hed[:,:,0]
hmin = np.percentile(hema_raw, 1)
hmax = np.percentile(hema_raw, 99)

# min max normalization, pixels with high hematoxylin will be approx 1 and others 0
hema_norm    = np.clip((hema_raw-hmin)/(hmax-hmin+1e-9), 0, 1).astype(np.float64)

# flipping the intensities so that dark nuclei appear on light bg
hema_display = 1.0 - hema_norm

# adding gaussian blur with a 2d kernel
# so every pixel becomes a weighted average of its neighbours with closer pixels having more weight
# so when plotting, you have continuous hills rather than spikes
hema_smooth  = filters.gaussian(hema_norm, sigma=GAUSS_SIGMA)

# skimage image thresholding
# otsu: automatic thresholding to find foreground/background separation
otsu   = threshold_otsu(hema_norm)
p_high = float(otsu)
p_low  = float(otsu * 0.45)
print(f"  τ_high={p_high:.3f}  τ_low={p_low:.3f}")



# computing PH through ripser / gudhi

small = sk_resize(hema_norm, (128,128), anti_aliasing=True).astype(np.float64)
small = (small-small.min())/(small.max()-small.min()+1e-9)


cc = gudhi.CubicalComplex(top_dimensional_cells=small)
cc.compute_persistence()

def get_dgm(cc, dim):
    arr = np.array(cc.persistence_intervals_in_dimension(dim), dtype=float)
    if arr.size==0: return np.zeros((0,2))
    return arr[np.isfinite(arr).all(axis=1)]

dgm1 = get_dgm(cc, 1)   # ALL H1 features
if len(dgm1):
    b1, d1 = dgm1[:,0], dgm1[:,1]
    pers   = d1 - b1
    print(f"  H1={len(b1)}  pers [{pers.min():.3f},{pers.max():.3f}]  "
          f"median={np.median(pers):.3f}")
else:
    b1=d1=pers=np.array([])


fig = plt.figure(figsize=(22,8))
fig.patch.set_facecolor("white")
gs = GridSpec(1,6,figure=fig,wspace=0.3,left=0.03,right=0.97,top=0.90,bottom=0.08)
ax_a = fig.add_subplot(gs[0,0])
ax_b = fig.add_subplot(gs[0,1])
ax_c = fig.add_subplot(gs[0,2], projection="3d")
gs_d = GridSpecFromSubplotSpec(2,1,subplot_spec=gs[0,3],hspace=0.08)
ax_d_top = fig.add_subplot(gs_d[0])
ax_d_bot = fig.add_subplot(gs_d[1])
ax_e = fig.add_subplot(gs[0,4])
ax_f = fig.add_subplot(gs[0,5])

# (a)
ax_a.imshow(img_rgb); ax_a.axis("off")
ax_a.set_title("(a) H&E", fontsize=11, fontweight="bold")

# (b)
ax_b.imshow(hema_display, cmap="gray", vmin=0, vmax=1)
ax_b.axis("off")
ax_b.set_title("(b) Hematoxylin color space", fontsize=11, fontweight="bold")

# (c) surface — planes BEHIND surface, outlines on top
step = 6
Yi = np.arange(0, CROP_SIZE, step)
Xi = np.arange(0, CROP_SIZE, step)
XX, YY = np.meshgrid(Xi, Yi)
ZZ = hema_smooth[::step, ::step]

x0p,x1p = float(Xi[0]),float(Xi[-1])
y0p,y1p = float(Yi[0]),float(Yi[-1])
Xp = np.array([[x0p,x1p],[x0p,x1p]])
Yp = np.array([[y0p,y0p],[y1p,y1p]])

# planes first (behind)
for z_val, col in [(p_low,"tomato"),(p_high,"dodgerblue")]:
    Zp = np.full_like(Xp, z_val, dtype=float)
    ax_c.plot_surface(Xp, Yp, Zp, color=col, alpha=0.50, linewidth=0)

# surface
ax_c.plot_surface(XX, YY, ZZ, cmap="viridis", linewidth=0,
                  antialiased=True, alpha=0.75)

# plane outlines last (always on top)
for z_val, col in [(p_high,"dodgerblue"),(p_low,"tomato")]:
    xs = [x0p,x1p,x1p,x0p,x0p]
    ys = [y0p,y0p,y1p,y1p,y0p]
    ax_c.plot(xs, ys, [z_val]*5, color=col, linewidth=2.5, zorder=10)

ax_c.set_zlim(0,1)
ax_c.view_init(elev=30, azim=210)
ax_c.set_axis_off()
ax_c.set_title("(c) Surface plot", fontsize=11, fontweight="bold")

# (d)
bin_high = ((hema_norm >= p_high)*255).astype(np.uint8)
bin_low  = ((hema_norm >= p_low )*255).astype(np.uint8)
for ax, bimg, bcol, tau, is_top in [
    (ax_d_top, bin_high, "dodgerblue", p_high, True),
    (ax_d_bot, bin_low,  "tomato",     p_low,  False),
]:
    ax.imshow(bimg, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.axis("off")
    for side in ["top","bottom","left","right"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_edgecolor(bcol)
        ax.spines[side].set_linewidth(3)
    if not is_top:
        ax.set_title(f"τ = {tau:.2f}", fontsize=8, color=bcol, pad=2)
ax_d_top.set_title("(d) Binarisations", fontsize=11, fontweight="bold", pad=6)

# (e) persistence diagram — all H1 features, median split
if len(b1) > 0:
    p_split = float(np.median(pers))   # median: natural noise/signal boundary
    pt_cols = ["tomato" if p >= p_split else "dodgerblue" for p in pers]

    lo = 0.0
    hi = max(d1.max(), b1.max()) + 0.02

    xx = np.linspace(lo, hi, 300)
    # blue: near diagonal (noise)
    ax_e.fill_between(xx, xx, np.minimum(xx+p_split, hi),
                      alpha=0.18, color="dodgerblue", zorder=1)
    # red: far from diagonal (signal)
    ax_e.fill_between(xx, np.minimum(xx+p_split, hi), hi,
                      alpha=0.18, color="tomato", zorder=1)
    ax_e.plot([lo,hi],[lo,hi], "k--", lw=0.9, zorder=2)
    ax_e.scatter(b1, d1, marker="^", c=pt_cols, s=20, alpha=0.80,
                 edgecolors="none", zorder=4)

    ax_e.set_xlim(lo, hi); ax_e.set_ylim(lo, hi)
    ax_e.tick_params(labelsize=8)
else:
    ax_e.text(0.5,0.5,"No H1 features",ha="center",va="center",
              transform=ax_e.transAxes)

ax_e.set_xlabel("Birth", fontsize=9)
ax_e.set_ylabel("Death", fontsize=9)
ax_e.set_title("(e) Persistence diagram\n(H₁ cycles)", fontsize=11, fontweight="bold")

# (f)
PALETTE = list(plt.cm.Set1.colors)+list(plt.cm.Set2.colors)
CLASS_COLORS={}
def get_color(label):
    if label not in CLASS_COLORS:
        CLASS_COLORS[label]=PALETTE[len(CLASS_COLORS)%len(PALETTE)]
    return CLASS_COLORS[label]

ax_f.imshow(img_rgb, alpha=0.45)
ax_f.set_xlim(0,CROP_SIZE); ax_f.set_ylim(CROP_SIZE,0)
ax_f.axis("off")
legend_handles=[]; plotted_labels=set(); drawn=0

for feat in features_gj:
    geom  = feat.get("geometry",{})
    props = feat.get("properties",{})
    label = ((props.get("classification") or {}).get("name")
             or props.get("label") or props.get("class")
             or props.get("objectType") or props.get("type") or "annotation")
    fc = get_color(label)
    gtype = geom.get("type","")
    rings = ([geom["coordinates"][0]] if gtype=="Polygon"
             else [p[0] for p in geom["coordinates"]] if gtype=="MultiPolygon"
             else [])
    for ring in rings:
        pts = np.array(ring,dtype=float)
        pts[:,0]-=col0; pts[:,1]-=row0
        if pts.ndim!=2 or pts.shape[0]<3: continue
        if pts[:,0].max()<0 or pts[:,0].min()>CROP_SIZE: continue
        if pts[:,1].max()<0 or pts[:,1].min()>CROP_SIZE: continue
        ax_f.add_patch(plt.Polygon(pts[:,:2],closed=True,fill=True,
                       facecolor=(*fc,0.25),edgecolor=fc,linewidth=1.5))
        drawn+=1
        if label not in plotted_labels:
            legend_handles.append(mpatches.Patch(facecolor=fc,label=label))
            plotted_labels.add(label)

print(f"  Drew {drawn} polygons")
if legend_handles:
    ax_f.legend(handles=legend_handles,fontsize=7,loc="upper right",framealpha=0.8)
ax_f.set_title("(f) Ground truth", fontsize=11, fontweight="bold")

fig.suptitle("Persistent Homology Pipeline — Subset1_Test_1",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig(OUT_PATH, dpi=150, bbox_inches="tight")
print(f"Saved → {OUT_PATH}")

  τ_high=0.326  τ_low=0.147
  H1=1207  pers [0.000,0.988]  median=0.082
  Drew 1 polygons
Saved → /Users/chanuka/Desktop/codespaces/ph_cancer/figure_5.png


### Version 4

In [55]:
"""Version 4"""

raw = tifffile.imread(TIFF_PATH)
if raw.ndim == 4:  raw = raw[0]
if raw.ndim == 3 and raw.shape[0] in (1,3,4): raw = np.moveaxis(raw, 0, -1)
if raw.shape[-1] == 4: raw = raw[..., :3]
if raw.dtype != np.uint8: raw = (raw / raw.max() * 255).astype(np.uint8)

H, W   = raw.shape[:2]
cy, cx = H // 2, W // 2
half   = CROP_SIZE // 2
img_rgb = raw[cy-half:cy+half, cx-half:cx+half]
print(f"  full image: {H}×{W}   crop: {img_rgb.shape}")

hed       = color.rgb2hed(img_rgb)
hema_raw  = hed[:, :, 0]
hmin, hmax = np.percentile(hema_raw, 1), np.percentile(hema_raw, 99)
hema_norm  = np.clip((hema_raw - hmin) / (hmax - hmin + 1e-9), 0, 1)


hema_display = 1.0 - hema_norm 
# adding smooth blur through a 2d gaussian kernal. so every pixel becomes a 
# weighted average of its neighbours with closer pixels having more weight
# so when plotting, you have continuous hills rather than spikes
hema_smooth  = filters.gaussian(hema_norm, sigma=GAUSS_SIGMA)

p_high = float(np.percentile(hema_norm, T_high))
p_low  = float(np.percentile(hema_norm, T_low))
print(f"  τ_high={p_high:.3f}  τ_low={p_low:.3f}")



# run ripser on the RAW image values (sublevel-set filtration).
# max dimensions is 1 
# H0: Connected components H1: Loops, Glands 

# Sublevel filtration starts from 0 → stroma enters first, nuclei last.
# H1 cycle: born when a ring of pixels first closes, dies when filled in.
# This is exactly what we want: gland lumens appear as H1 cycles.


print("Computing PH …")
small    = sk_resize(hema_norm, (128, 128), anti_aliasing=True).astype(np.float64)
result   = ripser(small, maxdim=1)
dgm0     = result["dgms"][0]
dgm1     = result["dgms"][1]

# Keep only finite points
dgm0 = dgm0[np.isfinite(dgm0).all(axis=1)]
dgm1 = dgm1[np.isfinite(dgm1).all(axis=1)]

# birth death pairs for H0
b0, d0 = dgm0[:,0], dgm0[:,1]

# birth death pairs for H1
b1, d1 = dgm1[:,0], dgm1[:,1]

print(f"  H0: {len(b0)}  H1: {len(b1)}")
if len(b1):
    print(f"  H1 birth [{b1.min():.3f}, {b1.max():.3f}]  "
          f"death [{d1.min():.3f}, {d1.max():.3f}]")




# dealing with the ground truth file - geojason

with open(GEOJSON_PATH) as fj:
    gj = json.load(fj)
features_gj = gj.get("features", [])
print(f"  {len(features_gj)} features")

# Gather all coordinates to understand the space
all_x, all_y = [], []
for feat in features_gj:
    geom  = feat.get("geometry", {})
    gtype = geom.get("type","")
    rings = []
    if gtype == "Polygon":      rings = [geom["coordinates"][0]]
    elif gtype == "MultiPolygon": rings = [p[0] for p in geom["coordinates"]]
    for ring in rings:
        pts = np.array(ring, dtype=float)
        all_x.extend(pts[:,0].tolist())
        all_y.extend(pts[:,1].tolist())

if all_x:
    xmin,xmax = min(all_x), max(all_x)
    ymin,ymax = min(all_y), max(all_y)
    print(f"  coords  x:[{xmin:.2f}, {xmax:.2f}]  y:[{ymin:.2f}, {ymax:.2f}]")
    print(f"  image   x:[0, {W}]  y:[0, {H}]")
    
    if xmax <= 1.5:
    
        def to_crop(ring):
            pts = np.array(ring, dtype=float)
            pts[:,0] = pts[:,0] * W - (cx - half)
            pts[:,1] = pts[:,1] * H - (cy - half)
            return pts
        print("  → normalised coords detected")
    else:
        # Pixel coords
        def to_crop(ring):
            pts = np.array(ring, dtype=float)
            pts[:,0] -= (cx - half)
            pts[:,1] -= (cy - half)
            return pts
        print("  → pixel coords detected")
else:
    def to_crop(ring):
        return np.array(ring, dtype=float)

PALETTE = list(plt.cm.Set1.colors) + list(plt.cm.Set2.colors)
CLASS_COLORS = {}
def get_color(label):
    if label not in CLASS_COLORS:
        CLASS_COLORS[label] = PALETTE[len(CLASS_COLORS) % len(PALETTE)]
    return CLASS_COLORS[label]



# plotting

fig = plt.figure(figsize=(22, 8))
fig.patch.set_facecolor("white")
gs = GridSpec(1, 6, figure=fig, wspace=0.3,
              left=0.03, right=0.97, top=0.90, bottom=0.08)

ax_a = fig.add_subplot(gs[0,0])
ax_b = fig.add_subplot(gs[0,1])
ax_c = fig.add_subplot(gs[0,2], projection="3d")
gs_d     = GridSpecFromSubplotSpec(2,1, subplot_spec=gs[0,3], hspace=0.08)
ax_d_top = fig.add_subplot(gs_d[0])
ax_d_bot = fig.add_subplot(gs_d[1])
ax_e = fig.add_subplot(gs[0,4])
ax_f = fig.add_subplot(gs[0,5])

# (a)
ax_a.imshow(img_rgb); ax_a.axis("off")
ax_a.set_title("(a) H&E", fontsize=11, fontweight="bold")

# (b)
ax_b.imshow(hema_display, cmap="gray", vmin=0, vmax=1)
ax_b.axis("off")
ax_b.set_title("(b) ", fontsize=11, fontweight="bold")

# (c) 3D surface
step = 4
Yi = np.arange(0, CROP_SIZE, step)
Xi = np.arange(0, CROP_SIZE, step)
XX, YY = np.meshgrid(Xi, Yi)
ZZ = hema_smooth[::step, ::step]

ax_c.plot_surface(XX, YY, ZZ, cmap="viridis",
                  linewidth=0, antialiased=False, alpha=0.88)

for z_val, col in [(p_high, "dodgerblue"), (p_low, "tomato")]:
    xs = [Xi[0], Xi[-1], Xi[-1], Xi[0]]
    ys = [Yi[0], Yi[0],  Yi[-1], Yi[-1]]
    verts = [list(zip(xs, ys, [z_val]*4))]
    plane = Poly3DCollection(verts, alpha=0.30,
                             facecolor=col, edgecolor=col, linewidth=0.5)
    ax_c.add_collection3d(plane)

ax_c.set_zlim(0, 1)
ax_c.view_init(elev=30, azim=-55)
ax_c.set_axis_off()
ax_c.set_title("(c) Surface plot", fontsize=11, fontweight="bold")

# (d) binarisations — white = nuclei active
bin_high = ((hema_norm >= p_high)*255).astype(np.uint8)
bin_low  = ((hema_norm >= p_low )*255).astype(np.uint8)

for ax, bimg, bcol, tau, is_top in [
    (ax_d_top, bin_high, "dodgerblue", p_high, True),
    (ax_d_bot, bin_low,  "tomato",     p_low,  False),
]:
    ax.imshow(bimg, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.axis("off")
    for side in ["top","bottom","left","right"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_edgecolor(bcol)
        ax.spines[side].set_linewidth(3)
    if not is_top:
        ax.set_title(f"τ = {tau:.2f}", fontsize=8, color=bcol, pad=2)

ax_d_top.set_title("(d) Binarisations", fontsize=11, fontweight="bold", pad=6)

if len(b1) > 0:
    pers  = d1 - b1
    p_med = np.median(pers)

    pt_colors = ["tomato" if p >= p_med else "dodgerblue" for p in pers]

    ax_e.scatter(b1, d1, marker="^", c=pt_colors, s=35, alpha=0.8, zorder=4)

    lo = min(b1.min(), d1.min()) - 0.02
    hi = max(b1.max(), d1.max()) + 0.02

    ax_e.plot([lo, hi], [lo, hi], "k--", lw=0.9, zorder=2)

    xx = np.linspace(lo, hi, 200)
    ax_e.fill_between(xx, xx,        xx + p_med, alpha=0.12, color="tomato",     zorder=1)
    ax_e.fill_between(xx, xx - 0.02, xx,         alpha=0.12, color="dodgerblue", zorder=1)

    ax_e.set_xlim(lo, hi)
    ax_e.set_ylim(lo, hi)
else:
    ax_e.text(0.5, 0.5, "No H1 features\n(try larger image or\ndifferent crop)",
              ha="center", va="center", transform=ax_e.transAxes, fontsize=9)

ax_e.set_xlabel("Birth", fontsize=9)
ax_e.set_ylabel("Death", fontsize=9)
ax_e.set_title("(e) Persistence diagram\n(H₁ cycles)", fontsize=11, fontweight="bold")

ax_f.imshow(img_rgb, alpha=0.45)
ax_f.set_xlim(0, CROP_SIZE)
ax_f.set_ylim(CROP_SIZE, 0)
ax_f.axis("off")

legend_handles = []
plotted_labels = set()
drawn = 0

for feat in features_gj:
    geom  = feat.get("geometry", {})
    props = feat.get("properties", {})
    label = (
        (props.get("classification") or {}).get("name")
        or props.get("label") or props.get("class")
        or props.get("objectType") or props.get("type") or "annotation"
    )
    fc    = get_color(label)
    gtype = geom.get("type","")
    rings = []
    if gtype == "Polygon":       rings = [geom["coordinates"][0]]
    elif gtype == "MultiPolygon": rings = [p[0] for p in geom["coordinates"]]

    for ring in rings:
        pts = to_crop(ring)
        if pts[:,0].max() < 0 or pts[:,0].min() > CROP_SIZE: continue
        if pts[:,1].max() < 0 or pts[:,1].min() > CROP_SIZE: continue
        poly = plt.Polygon(pts[:,:2], closed=True, fill=True,
                           facecolor=(*fc, 0.25), edgecolor=fc, linewidth=1.5)
        ax_f.add_patch(poly)
        drawn += 1
        if label not in plotted_labels:
            legend_handles.append(mpatches.Patch(facecolor=fc, label=label))
            plotted_labels.add(label)

print(f"  Drew {drawn} polygons in ground-truth panel")
if legend_handles:
    ax_f.legend(handles=legend_handles, fontsize=7, loc="upper right", framealpha=0.8)
ax_f.set_title("(f) Ground truth", fontsize=11, fontweight="bold")

fig.suptitle("Persistent Homology Pipeline — Subset1_Test_1",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig(OUT_PATH, dpi=150, bbox_inches="tight")
print(f"\nSaved → {OUT_PATH}")

  full image: 90000×46080   crop: (1024, 1024, 3)
  τ_high=0.091  τ_low=0.014
Computing PH …
  H0: 127  H1: 73
  H1 birth [1.041, 1.780]  death [1.073, 1.790]
  72 features
  coords  x:[5260.00, 39110.00]  y:[32100.00, 85890.00]
  image   x:[0, 46080]  y:[0, 90000]
  → pixel coords detected
  Drew 2 polygons in ground-truth panel

Saved → /Users/chanuka/Desktop/codespaces/ph_cancer/figure_5.png
